# Exercise 8: what can we do with BERT?

A review needs one label. Named entity recognition needs a label for each token. Extractive question answering needs a start and an end position. We will use three independently fine-tuned BERT models and build the decisions that turn their scores into useful outputs.

| Task | Model output | Decision |
|---|---|---|
| Product-review sentiment | `(B, 5)` logits | One star rating per review |
| Question answering | Two `(B, L)` score tensors | A context span, or no answer |
| Named entity recognition | `(B, L, 9)` logits | Token labels grouped into text spans |

**B** is batch size, **L** is the padded token-sequence length. The models share an architecture, not their fine-tuned weights. Int this exercise we will perform inference, parameters are not trained here.


### Setup

Use PyTorch and Transformers 4.x. If needed, run `%pip install torch "transformers>=4.46,<5"` in a separate cell and restart the kernel. Matplotlib is optional.

Cache the three checkpoints before the session. Set `CACHE_ONLY=True` to require cached files. The notebook loads one model at a time and supports CPU. Repository revisions are pinned for reproducible examples. Fast tokenizers provide the character offsets used below.


In [ ]:
import gc
import math
from pathlib import Path

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForTokenClassification

torch.manual_seed(42)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = Path("hf_cache")
CACHE_ONLY = False
MAX_LENGTH = 128

MODEL_INFO = {
            'sentiment': 
                {'name': 'nlptown/bert-base-multilingual-uncased-sentiment', 
                'revision': '8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d'}, 
            'qa':
                {'name': 'deepset/bert-base-cased-squad2', 
                'revision': 'd378e310c8000d824954c1e76c943a0581b49f0c'}, 
            'ner': 
                {'name': 'dslim/bert-base-NER', 
                'revision': 'd1a3e8f13f8c3566299d95fcfc9a8d2382a9affc'}
            }
MODEL_CLASSES = {"sentiment": AutoModelForSequenceClassification,
                 "qa": AutoModelForQuestionAnswering,
                 "ner": AutoModelForTokenClassification}

def load_checkpoint(task):
    info = MODEL_INFO[task]
    options = dict(revision=info["revision"], cache_dir=str(CACHE_DIR), local_files_only=CACHE_ONLY)
    tokenizer = AutoTokenizer.from_pretrained(info["name"], use_fast=True, **options)
    model = MODEL_CLASSES[task].from_pretrained(info["name"], **options).to(DEVICE)
    model.eval()
    assert tokenizer.is_fast
    print("Loaded:", info["name"], "| revision:", info["revision"][:12])
    return tokenizer, model

def close(actual, expected, atol=1e-5, rtol=1e-4):
    actual = torch.as_tensor(actual).detach().cpu()
    expected = torch.as_tensor(expected, dtype=actual.dtype).detach().cpu()
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)

def must_raise(error_type, callback):
    try:
        callback()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__, "| device:", DEVICE)
sentiment_tokenizer, sentiment_model = load_checkpoint("sentiment")


### From text to tensors

BERT reads token IDs, not strings. Its tokenizer can split a word into several pieces. In a padded batch, `attention_mask` distinguishes real input positions from padding. Special tokens such as `[CLS]` and `[SEP]` still have mask value 1.

The tokenizer accepts `return_tensors="pt"`, `padding=True`, `truncation=True`, and `max_length`. Its output is a dictionary-like batch. The examples fit the chosen short-input limits, handling long documents with overlapping windows is outside this exercise.


## Task 1: Build a batch BERT can read

Tokenize a list of texts into a padded, length-limited PyTorch batch containing the inputs needed by BERT.


In [ ]:
def tokenize_batch(tokenizer, texts, max_length=MAX_LENGTH):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# Two different-length texts become one rectangular batch.
token_examples = ["Excellent!", "This uncharacteristically overpriced device is not bad."]
encoded = tokenize_batch(sentiment_tokenizer, token_examples)
print(f"{len(token_examples)} texts -> input_ids shape {tuple(encoded['input_ids'].shape)}")
for row, text in enumerate(token_examples):
    tokens = sentiment_tokenizer.convert_ids_to_tokens(encoded["input_ids"][row].tolist())
    mask = encoded["attention_mask"][row].tolist()
    print(f"\n{text!r}")
    print("Tokens:", " ".join(tokens))
    print("Mask:  ", " ".join(map(str, mask)))
    print(f"Keep {sum(mask)} positions; ignore {len(mask) - sum(mask)} padding positions.")
print("\nThe ## pieces belong to the preceding word; [PAD] fills the shorter row.")

assert encoded["input_ids"].shape == encoded["attention_mask"].shape
assert encoded["input_ids"].dtype == torch.long
assert (encoded["attention_mask"] == 0).any()
assert encoded["input_ids"].shape[1] <= MAX_LENGTH
assert all(int(value) == sentiment_tokenizer.pad_token_id
           for value in encoded["input_ids"][encoded["attention_mask"] == 0])


### Inference and output heads

`model(**inputs)` returns a structured output. Classification heads expose `.logits`; the QA head exposes `.start_logits` and `.end_logits`. Only `input_ids`, `attention_mask`, and `token_type_ids` are model inputs here. Character offsets are metadata for decoding.

Evaluation mode controls layers such as dropout. Disabling gradients avoids tracking computations that will not be backpropagated. These are separate responsibilities.


## Task 2: Run inference once for a whole batch

Run the model in evaluation mode without gradients, on its device, while keeping decoding metadata out of model inputs.


In [ ]:
def run_model(model, batch):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# Does batching a short text with a longer one change its prediction?
batched_output = run_model(sentiment_model, encoded)
batch_logits = batched_output.logits.detach().cpu()
single_output = run_model(sentiment_model, tokenize_batch(sentiment_tokenizer, [token_examples[0]]))
single_logits = single_output.logits[0].detach().cpu()
print(f"Input: {tuple(encoded['input_ids'].shape)} token IDs -> output: {tuple(batch_logits.shape)} class scores")
print(f"\nScores for {token_examples[0]!r}:")
print("Alone:   ", [round(x, 3) for x in single_logits.tolist()])
print("In batch:", [round(x, 3) for x in batch_logits[0].tolist()])
print(f"Largest difference: {(single_logits - batch_logits[0]).abs().max().item():.2e}")
print("\nThe attention mask lets us add padding without changing this prediction.")
print(f"Evaluation mode: {not sentiment_model.training}; tracking gradients: {batched_output.logits.requires_grad}")

close(single_logits, batch_logits[0], atol=1e-4)
assert batch_logits.shape == (2, sentiment_model.config.num_labels)
assert not batched_output.logits.requires_grad and not sentiment_model.training
# Offsets help decoding, but must not be forwarded to BERT.
with_offsets = sentiment_tokenizer(token_examples[0], return_offsets_mapping=True, return_tensors="pt")
metadata_output = run_model(sentiment_model, with_offsets)
close(metadata_output.logits[0], single_logits, atol=1e-4)


### Part I: one distribution for a review

This checkpoint predicts five star-rating classes for product reviews. A softmax score is the model's mass on one label, it is not a measured probability that the decision is correct. Keep the complete distribution when comparing uncertain or conflicting reviews.


## Task 3: Turn class scores into decisions

Return probabilities, predicted class IDs, and their scores for logits with an arbitrary number of leading dimensions.


In [ ]:
def classify_scores(logits):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
reviews = [
    "The product is excellent.",
    "The product is not excellent.",
    "The product looks excellent, but it arrived broken and does not work.",
]
review_batch = tokenize_batch(sentiment_tokenizer, reviews)
review_logits = run_model(sentiment_model, review_batch).logits.detach().cpu()
review_probabilities, review_ids, review_scores = classify_scores(review_logits)
labels = [sentiment_model.config.id2label[i] for i in range(review_logits.shape[-1])]
print("Read each row as: raw scores -> distribution -> chosen label.\n")
for text, logits, probabilities, label_id, score in zip(
        reviews, review_logits, review_probabilities, review_ids, review_scores):
    print(text)
    print("  Logits: ", " ".join(f"{x:6.2f}" for x in logits.tolist()))
    print("  Classes:", " ".join(f"{i + 1:5d}★" for i in range(len(labels))))
    print("  Mass:   ", " ".join(f"{p:6.1%}" for p in probabilities.tolist()))
    print(f"  Decision: {labels[int(label_id)]} (class mass {float(score):.1%})\n")
print("All three contain 'excellent'. Compare how the surrounding words change the decision.")

close(review_probabilities.sum(-1), torch.ones(len(reviews)))
assert review_ids.shape == review_scores.shape == (len(reviews),)
assert torch.equal(review_ids, review_logits.argmax(-1))
# The same function also handles one distribution per token.
p, ids, s = classify_scores(torch.zeros(2, 4, 3))
assert p.shape == (2, 4, 3) and ids.shape == s.shape == (2, 4)
close(p.sum(-1), torch.ones(2, 4))


### Scores and certainty

Temperature rescales logits before softmax: `softmax(logits / T)`, for T > 0. It can change how concentrated a distribution looks while keeping its highest-scoring class. Here T is an experiment, not a calibration parameter fitted on labeled validation data.


## Task 4: Change certainty without changing the predicted label

Return class probabilities after applying a positive temperature to the logits.


In [ ]:
def temperature_probabilities(logits, temperature):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# Change only temperature: no new model call and no training.
selected_review = 1
print(reviews[selected_review])
print("\nTemperature   Predicted label   Largest class mass")
temperatures = [0.5, 1.0, 2.0]
temperature_rows = []
for value in temperatures:
    p = temperature_probabilities(review_logits[selected_review], value)
    temperature_rows.append(p)
    bar = "█" * round(float(p.max()) * 30)
    print(f"{value:>7.1f}       {labels[int(p.argmax())]:<12}      {float(p.max()):5.1%}  {bar}")
    close(p.sum(), 1.)
    assert int(p.argmax()) == int(review_ids[selected_review])
print("\nLower T sharpens the distribution; higher T flattens it.")
print("The chosen label stays the same. A more confident-looking score need not be more correct.")

assert temperature_rows[0].max() >= temperature_rows[-1].max()
must_raise(ValueError, lambda: temperature_probabilities(torch.ones(3), 0))


### Part II: an answer is a span

The QA checkpoint was fine-tuned on SQuAD 2.0, which includes unanswerable questions. It scores each token as a possible start and end. A valid answer must lie in the context, start before it ends, and respect a length limit. Choosing the best start and end independently can violate these constraints.

A fast tokenizer's `sequence_ids(0)` marks question tokens with 0 and context tokens with 1, special tokens have `None`. Character offsets are half-open intervals in their respective original strings. Only offsets for the context may be used to extract an answer from the passage.


In [ ]:
del sentiment_model, batched_output, single_output, metadata_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
qa_tokenizer, qa_model = load_checkpoint("qa")

context = (
    "Maya Chen signed the research agreement on 14 March 2025. "
    "The agreement allocated two million euros to the Riverside project. "
    "The signing ceremony took place in Berlin. "
    "Alex Rivera attended the ceremony as a witness."
)
question = "How much funding did the agreement allocate?"
qa_batch = qa_tokenizer(question, context, return_tensors="pt", return_offsets_mapping=True,
                        truncation="only_second", max_length=256)
qa_output = run_model(qa_model, qa_batch)
start_logits = qa_output.start_logits[0].detach().cpu()
end_logits = qa_output.end_logits[0].detach().cpu()
qa_offsets = qa_batch["offset_mapping"][0].tolist()
qa_sequence_ids = qa_batch.sequence_ids(0)
print("Fictional passage:", context)
print("Question:", question)
print("Start/end score shapes:", tuple(start_logits.shape), tuple(end_logits.shape))


## Task 5: Find the best legal answer span

Find the highest-scoring context span under the start, end, and maximum-length constraints, or return None if no span is valid.


In [ ]:
def best_span(start_logits, end_logits, context_mask, max_answer_length=20):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# The strongest start and end can form an illegal span.
toy_tokens = ["[CLS]", "Maya", "Chen", "in", "Berlin"]
toy_start = torch.tensor([0., 9., 1., 0., 0.])
toy_end = torch.tensor([0., 0., 8., 0., 10.])
toy_mask = [False, True, True, True, True]
span = best_span(toy_start, toy_end, toy_mask, max_answer_length=2)
print("Tokens:", " | ".join(toy_tokens))
print("Start scores:", toy_start.tolist())
print("End scores:  ", toy_end.tolist())
print("\nIndependent maxima: 'Maya Chen in Berlin' (score 19; too long)")
print(f"Best legal span:    {' '.join(toy_tokens[span[0]:span[1]+1])!r} (score {span[2]:.0f}; at most 2 tokens)")
assert span == (1, 2, 17.)
assert best_span(toy_start, toy_end, [False]*5) is None
assert best_span(toy_start, toy_end, [False, True, False, False, True], 5) == (4, 4, 10.)
must_raise(ValueError, lambda: best_span(toy_start, toy_end, toy_mask, 0))

context_mask = [sid == 1 and end > start for sid, (start, end) in zip(qa_sequence_ids, qa_offsets)]
first, last, joint_score = best_span(start_logits, end_logits, context_mask)
print(f"\nReal question: {question}")
print(f"Best legal answer: {context[qa_offsets[first][0]:qa_offsets[last][1]]!r}")
print("We choose the best start/end PAIR that satisfies all the constraints.")
assert first <= last and last-first+1 <= 20
assert all(context_mask[i] for i in range(first, last+1))


### Answer or abstain

For this BERT QA head, the `[CLS]` start-plus-end score represents a no-answer alternative. Compare it with the best non-null span score. We abstain when `null score − best span score > threshold`.

This is an explicit decision rule. Its margin is not a calibrated answer probability. A threshold of zero is a classroom default, a deployment threshold would need labeled validation data. The passage below is the source of the answer, not external knowledge about the named people.


## Task 6: Decode an answer and allow no answer

Return the exact answer substring and character span, or abstain according to the null-score margin.


In [ ]:
def decode_answer(context, offsets, sequence_ids, start_logits, end_logits,
                  cls_index=0, null_threshold=0.0, max_answer_length=20):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# Hold the candidate answer fixed; change only the no-answer score.
toy_context = "Maya Chen"
toy_offsets = [[0, 0], [0, 4], [5, 9], [0, 0]]
toy_sequences = [None, 1, 1, None]
print("Controlled example: candidate answer is 'Maya Chen', span score = 16.")
print("Null score   Margin (null - span)   Decision at threshold 0")
for cls_score in [0., 10.]:
    decoded = decode_answer(toy_context, toy_offsets, toy_sequences,
                            torch.tensor([cls_score, 8., 0., 0.]),
                            torch.tensor([cls_score, 0., 8., 0.]))
    decision = repr(decoded["answer"]) if decoded["answer"] is not None else "NO ANSWER"
    print(f"{2 * cls_score:>6.0f}             {decoded['null_margin']:>+5.0f}              {decision}")
    assert (decoded["answer"] is None) == (cls_score == 10.)
    if decoded["answer"] is not None:
        assert (decoded["start"], decoded["end"]) == (0, 9)
print("A legal span exists in both cases. The margin decides whether to return it.\n")

questions = ["How much funding did the agreement allocate?", "Who signed the research agreement?",
             "What is Maya Chen's shoe size?"]
passage_answers = ["two million euros", "Maya Chen", None]
qa_results = []
for text, passage_answer in zip(questions, passage_answers):
    enc = qa_tokenizer(text, context, return_offsets_mapping=True, return_tensors="pt",
                       truncation="only_second", max_length=256)
    output = run_model(qa_model, enc)
    start, end = output.start_logits[0].detach().cpu(), output.end_logits[0].detach().cpu()
    decoded = decode_answer(context, enc["offset_mapping"][0].tolist(), enc.sequence_ids(0), start, end)
    qa_results.append(decoded)
    print(text)
    print(f"  Passage answer: {passage_answer!r}; model answer: {decoded['answer']!r}")
    print(f"  Null margin: {decoded['null_margin']:+.2f} -> {'abstain' if decoded['answer'] is None else 'return span'}")
    if decoded["answer"] is not None:
        assert context[decoded["start"]:decoded["end"]] == decoded["answer"]
print("\nCompare passage and model answers: a correct decoder does not guarantee a correct prediction.")


### Part III: one distribution per token

NER labels use BIO notation: `B-PER` begins a person, `I-PER` continues a person, and `O` is outside an entity. The same pattern applies to ORG, LOC, and MISC. An entity may span several words or several pieces of one word.

Our decoder starts a new entity at every B tag. An orphan I tag, or an I tag with a different type, starts a new entity as a recovery policy. Special tokens have empty character offsets. Reconstruct strings with character offsets, preserving the original spelling and spaces.


In [ ]:
del qa_model, qa_output, output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
ner_tokenizer, ner_model = load_checkpoint("ner")
ner_text = "Maya Chen met Alex Rivera in Darmstadt while working for Microsoft."
ner_batch = ner_tokenizer(ner_text, return_offsets_mapping=True, return_tensors="pt")
ner_logits = run_model(ner_model, ner_batch).logits[0].detach().cpu()
ner_probabilities, ner_ids, ner_scores = classify_scores(ner_logits)
ner_labels = [ner_model.config.id2label[int(i)] for i in ner_ids]
ner_offsets = ner_batch["offset_mapping"][0].tolist()
ner_tokens = ner_tokenizer.convert_ids_to_tokens(ner_batch["input_ids"][0].tolist())
print("Text:", ner_text)
print("Token distributions:", tuple(ner_logits.shape))
print(f"{'Token':<16} {'Label':<9} {'Offset':<12} {'Class mass':<10}")
for token, label, offset, score in zip(ner_tokens, ner_labels, ner_offsets, ner_scores):
    print(f"{token:<16} {label:<9} {str(offset):<12} {float(score):.3f}")


## Task 7: Reconstruct entities from BIO predictions

Group token-level BIO predictions into exact entity substrings and character spans under the stated boundary and recovery rules.


In [ ]:
def merge_bio(text, labels, offsets):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# Pieces of one word merge; a new B tag starts a separate entity.
toy_text = "Maya Alex in Berlin"
toy_labels = ["O", "B-PER", "I-PER", "B-PER", "O", "I-LOC", "O"]
toy_offsets = [[0, 0], [0, 2], [2, 4], [5, 9], [10, 12], [13, 19], [0, 0]]
toy_entities = merge_bio(toy_text, toy_labels, toy_offsets)
print("Before: token pieces + BIO labels")
print(" | ".join(f"{toy_text[a:b] or '[special]'}: {label}"
                 for label, (a, b) in zip(toy_labels, toy_offsets)))
print("\nAfter: entities recovered from the original text")
for entity in toy_entities:
    print(f"  {entity['type']}  {entity['text']!r}  [{entity['start']}:{entity['end']}]")
print("\nMa + ya -> Maya; B-PER on Alex keeps it separate; orphan I-LOC starts Berlin.")
assert [(e["type"], e["text"]) for e in toy_entities] == [("PER", "Maya"), ("PER", "Alex"), ("LOC", "Berlin")]
assert merge_bio("plain text", ["O", "O"], [[0, 5], [6, 10]]) == []
assert [e["type"] for e in merge_bio("AB", ["I-PER", "I-ORG"], [[0, 1], [1, 2]])] == ["PER", "ORG"]

entities = merge_bio(ner_text, ner_labels, ner_offsets)
print("\nReal model: subword tokens become readable entities")
print("; ".join(f"{e['text']} ({e['type']})" for e in entities))
print("Character offsets preserve the original spelling and spaces.")
for entity in entities:
    assert ner_text[entity["start"]:entity["end"]] == entity["text"]


## Task 8: Build an entity-extraction function

Combine tokenization, inference, token classification, and BIO decoding into an entity extractor for one short text.


In [ ]:
def extract_entities(text, tokenizer, model, max_length=MAX_LENGTH):
    ### Your Solution starts here ###
    raise NotImplementedError
    ### Your Solution ends here ###


In [ ]:
# One call now performs the full text -> entities workflow.
contrasts = [
    "Apple opened an office in Berlin.",
    "I ate an apple in Berlin.",
    "Maya Chen works for Microsoft in Düsseldorf.",
]
for text in contrasts:
    predicted_entities = extract_entities(text, ner_tokenizer, ner_model)
    # Insert entity labels into the original sentence, working from right to left.
    annotated = text
    for entity in reversed(predicted_entities):
        start, end = entity["start"], entity["end"]
        assert text[start:end] == entity["text"]
        annotated = annotated[:start] + f"[{entity['text']} | {entity['type']}]" + annotated[end:]
    print(f"Input:  {text}")
    print(f"Output: {annotated}\n")
print("Compare Apple/apple: does the context describe a company or a fruit?")
print("Düsseldorf shows why we recover spelling from the original text.")

assert extract_entities(ner_text, ner_tokenizer, ner_model) == entities
must_raise(ValueError, lambda: extract_entities("Berlin " * 150, ner_tokenizer, ner_model, max_length=16))

# Change this sentence to try the complete extractor yourself.
CLASS_TEXT = "Alex Rivera visited Paris while working for Microsoft."
print("\nYour turn:", CLASS_TEXT)
print(extract_entities(CLASS_TEXT, ner_tokenizer, ner_model))


### Model sources

- [nlptown multilingual review sentiment](https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment): five rating classes.
- [deepset BERT SQuAD 2.0](https://huggingface.co/deepset/bert-base-cased-squad2): extractive QA with a no-answer alternative.
- [dslim BERT NER](https://huggingface.co/dslim/bert-base-NER): BIO entity labels.
- [Hugging Face QA preprocessing](https://huggingface.co/docs/transformers/v4.57.1/tasks/question_answering): token offsets and context positions.
